## Combine, Clean & Status
Scans `RESULTS/` across all sizes and trials, reports failures/missing,
combines successful results into `COMBINED/`, applies quality filters,
and produces per-trial stats + per-size and overall MLPreprocessing visualizations.

In [1]:
import sys, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from joblib import Parallel, delayed

THERMOIFT_SRC = Path("../../thermoift/src").resolve()
if str(THERMOIFT_SRC) not in sys.path:
    sys.path.insert(0, str(THERMOIFT_SRC))

from thermoift import MLPreprocessing

In [2]:
RESULTS_DIR  = Path("RESULTS")
COMBINED_DIR = Path("COMBINED")
OUTPUT_DIR   = Path("OUTPUT")
IFT_SUBPATH  = "CSV/InterfacialProperties/feed_1_interfacial_results.csv"

# Number of parallel workers — reads SLURM_CPUS_PER_TASK when running on a cluster
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

# Quality filters (same as PRE_ML)
MAX_VAPOR_DENSITY  = 400    # kg/m3 — above this is unphysical
MIN_IFT_THICKNESS  = 0.0    # nm   — negative thickness is unphysical
MAX_IFT_THICKNESS  = 7.5    # nm   — above this is unphysical / near-critical artifact
MIN_GAMMA          = 0.05   # mN/m — near-zero IFT is unphysical (near-critical artifact)

COMPONENTS = [
    "carbon dioxide", "hydrogen", "argon", "nitrogen",
    "methane", "oxygen", "carbon monoxide", "hydrogen sulfide"
]
TARGET   = "gamma"
Z_COLS   = [f"z_{c}" for c in COMPONENTS]

print(f"N_JOBS = {N_JOBS}")

N_JOBS = 256


In [3]:
def run_ml_plots(df, out_folder, label=""):
    """Run all 7 MLPreprocessing plots into out_folder."""
    out_folder = Path(out_folder).resolve()
    out_folder.mkdir(parents=True, exist_ok=True)

    z_cols   = [c for c in Z_COLS if c in df.columns]
    features = ["temperature", "pressure"] + z_cols
    prep     = MLPreprocessing(df=df, features=features, target=TARGET)

    folder_str = str(out_folder)

    prep.plot_scatter("temperature", TARGET, "pressure",
                      save_path="gamma_vs_T",         folder=folder_str)
    plt.close("all")
    prep.plot_scatter("pressure",    TARGET, "temperature",
                      save_path="gamma_vs_P",         folder=folder_str)
    plt.close("all")
    prep.plot_scatter("liquid_density",       TARGET, "temperature",
                      save_path="gamma_vs_rhoL",      folder=folder_str)
    plt.close("all")
    prep.plot_scatter("vapor_density",        TARGET, "temperature",
                      save_path="gamma_vs_rhoV",      folder=folder_str)
    plt.close("all")
    prep.plot_scatter("interfacial_thickness", TARGET, "temperature",
                      save_path="gamma_vs_thickness", folder=folder_str)
    plt.close("all")

    from thermoift import PLOT_SETTINGS as ps

    fig, ax = prep.plot_histogram(TARGET, bins=50, save_path=None)
    ps.save_plot(fig, "gamma_distribution", folder=folder_str)
    plt.close("all")

    fig, ax = prep.plot_phase_envelope(group_by="temperature", value_col=TARGET, save_path=None)
    ps.save_plot(fig, "gamma_phase_envelope", folder=folder_str)
    plt.close("all")

    print(f"  Plots → {out_folder}")

In [4]:
def _process_trial(size, trial, task_map, combined_dir, ift_subpath,
                   min_ift, max_ift, min_gamma, max_rhov):
    """Read, filter, and write one (size, trial) task group. Returns stats dict or None."""
    import pandas as pd
    from pathlib import Path

    out_dir  = Path(combined_dir) / size
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{trial}.csv"

    dfs = []
    for task_id, folder in sorted(task_map.items()):
        try:
            df = pd.read_csv(Path(folder) / ift_subpath)
            df.insert(0, "task_id", task_id)
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: {Path(folder).name}: {e}", flush=True)

    if not dfs:
        return None

    raw = pd.concat(dfs, ignore_index=True)

    mask_nan        = raw["gamma"].isna() | raw["interfacial_thickness"].isna()
    mask_thick_low  = raw["interfacial_thickness"] <= min_ift
    mask_thick_high = raw["interfacial_thickness"] > max_ift
    mask_gamma_low  = raw["gamma"] < min_gamma
    mask_rhoV       = raw["vapor_density"] >= max_rhov

    valid = ~mask_nan
    n_nan        = int(mask_nan.sum())
    n_thick_low  = int((valid & mask_thick_low).sum())
    n_thick_high = int((valid & ~mask_thick_low & mask_thick_high).sum())
    n_gamma_low  = int((valid & ~mask_thick_low & ~mask_thick_high & mask_gamma_low).sum())
    n_rhoV       = int((valid & ~mask_thick_low & ~mask_thick_high & ~mask_gamma_low & mask_rhoV).sum())
    n_raw        = len(raw)

    clean = raw[valid & ~mask_thick_low & ~mask_thick_high & ~mask_gamma_low & ~mask_rhoV]
    n_clean = len(clean)
    clean.to_csv(out_path, index=False)

    return {
        "size":          size,
        "trial":         trial,
        "tasks":         len(task_map),
        "raw_rows":      n_raw,
        "nan_gamma":     n_nan,
        "bad_thick_low": n_thick_low,
        "bad_thick_hi":  n_thick_high,
        "bad_gamma_low": n_gamma_low,
        "bad_rhoV":      n_rhoV,
        "clean_rows":    n_clean,
        "pct_clean":     round(100 * n_clean / n_raw, 1) if n_raw else 0.0,
    }


def _plot_one_size(size, combined_dir_str, thermoift_src_str, z_cols, target):
    """Generate all 7 MLPreprocessing plots for one size. Runs in a loky worker."""
    import sys, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    from pathlib import Path

    thermoift_src = Path(thermoift_src_str)
    if str(thermoift_src) not in sys.path:
        sys.path.insert(0, str(thermoift_src))
    from thermoift import MLPreprocessing, PLOT_SETTINGS as ps

    combined_dir = Path(combined_dir_str)
    csvs = sorted((combined_dir / size).glob("trial_*.csv"))
    if not csvs:
        return f"{size}: no combined CSVs — skipping"

    df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    out_folder = (combined_dir / size / "PLOTS").resolve()
    out_folder.mkdir(parents=True, exist_ok=True)
    folder_str = str(out_folder)

    z = [c for c in z_cols if c in df.columns]
    prep = MLPreprocessing(df=df, features=["temperature", "pressure"] + z, target=target)

    prep.plot_scatter("temperature", target, "pressure",          save_path="gamma_vs_T",         folder=folder_str); plt.close("all")
    prep.plot_scatter("pressure",    target, "temperature",       save_path="gamma_vs_P",         folder=folder_str); plt.close("all")
    prep.plot_scatter("liquid_density",        target, "temperature", save_path="gamma_vs_rhoL",  folder=folder_str); plt.close("all")
    prep.plot_scatter("vapor_density",         target, "temperature", save_path="gamma_vs_rhoV",  folder=folder_str); plt.close("all")
    prep.plot_scatter("interfacial_thickness",  target, "temperature", save_path="gamma_vs_thickness", folder=folder_str); plt.close("all")

    fig, ax = prep.plot_histogram(target, bins=50, save_path=None)
    ps.save_plot(fig, "gamma_distribution",   folder=folder_str); plt.close("all")

    fig, ax = prep.plot_phase_envelope(group_by="temperature", value_col=target, save_path=None)
    ps.save_plot(fig, "gamma_phase_envelope", folder=folder_str); plt.close("all")

    return f"{size}: {len(df):,} rows from {len(csvs)} trials"

### Discover expected sizes and trials from OUTPUT/

In [5]:
expected = {}   # (size, trial) -> n_compositions
if OUTPUT_DIR.exists():
    for size_dir in sorted(OUTPUT_DIR.iterdir()):
        if not size_dir.is_dir():
            continue
        for csv in sorted(size_dir.glob("trial_*.csv")):
            trial = csv.stem
            n = sum(1 for _ in csv.open()) - 1
            expected[(size_dir.name, trial)] = n

print(f"Expected: {len(expected)} trials across {len({s for s,_ in expected})} sizes")
for size in sorted({s for s,_ in expected}):
    trials = sorted(t for s,t in expected if s == size)
    print(f"  {size}: {len(trials)} trials, {expected[(size, trials[0])]} compositions/trial")

Expected: 80 trials across 4 sizes
  N025: 20 trials, 25 compositions/trial
  N050: 20 trials, 50 compositions/trial
  N075: 20 trials, 75 compositions/trial
  N100: 20 trials, 100 compositions/trial


### Scan RESULTS/ and classify tasks

In [6]:
failures  = {}
missing   = {}
successes = {}

for (size, trial), n_expected in sorted(expected.items()):
    trial_dir = RESULTS_DIR / size / trial

    if not trial_dir.exists():
        missing[(size, trial)] = list(range(n_expected))
        continue

    by_task = defaultdict(list)
    for folder in trial_dir.iterdir():
        if not folder.is_dir():
            continue
        parts = folder.name.rsplit("_", 1)
        if len(parts) == 2 and parts[1].isdigit():
            by_task[int(parts[1])].append(folder)

    trial_failures  = []
    trial_missing   = []
    trial_successes = {}

    for task_id in range(n_expected):
        folders = by_task.get(task_id, [])
        if not folders:
            trial_missing.append(task_id)
        else:
            ok = [f for f in folders if (f / IFT_SUBPATH).exists()]
            if not ok:
                trial_failures.append(task_id)
            else:
                best = sorted(ok, key=lambda f: int(f.name.rsplit("_", 1)[0]))[-1]
                trial_successes[task_id] = best

    if trial_failures:
        failures[(size, trial)] = trial_failures
    if trial_missing:
        missing[(size, trial)] = trial_missing
    if trial_successes:
        successes[(size, trial)] = trial_successes

n_success = sum(len(v) for v in successes.values())
n_fail    = sum(len(v) for v in failures.values())
n_miss    = sum(len(v) for v in missing.values())
print(f"Successful tasks : {n_success}")
print(f"Failed tasks     : {n_fail}   (ran but produced no output)")
print(f"Missing tasks    : {n_miss}  (never ran)")

Successful tasks : 5000
Failed tasks     : 0   (ran but produced no output)
Missing tasks    : 0  (never ran)


### What is missing / failed

In [7]:
if not failures and not missing:
    print("All tasks completed successfully — nothing missing.")
else:
    if missing:
        print("=== Never ran (no result folder) ===")
        for (size, trial), task_ids in sorted(missing.items()):
            if len(task_ids) == expected[(size, trial)]:
                print(f"  {size}/{trial}  →  entire trial not submitted yet")
            else:
                print(f"  {size}/{trial}  →  {len(task_ids)} tasks never ran: {task_ids}")

    if failures:
        print("\n=== Ran but failed (no CSV output) ===")
        for (size, trial), task_ids in sorted(failures.items()):
            print(f"  {size}/{trial}  →  {len(task_ids)} failed tasks: {task_ids}")

All tasks completed successfully — nothing missing.


### Combine and clean — with per-trial statistics

In [8]:
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Combining {len(successes)} trial groups with {N_JOBS} parallel workers...")

results = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(_process_trial)(
        size, trial, task_map,
        COMBINED_DIR, IFT_SUBPATH,
        MIN_IFT_THICKNESS, MAX_IFT_THICKNESS, MIN_GAMMA, MAX_VAPOR_DENSITY
    )
    for (size, trial), task_map in sorted(successes.items())
)

trial_stats = [r for r in results if r is not None]
stats_df = pd.DataFrame(trial_stats).sort_values(["size", "trial"]).reset_index(drop=True)
print(f"Written {len(trial_stats)} combined CSVs to {COMBINED_DIR}/")

Combining 80 trial groups with 256 parallel workers...


Written 80 combined CSVs to COMBINED/


### Per-trial cleaning statistics

In [9]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
print(stats_df.to_string(index=False))

size    trial  tasks  raw_rows  nan_gamma  bad_thick_low  bad_thick_hi  bad_gamma_low  bad_rhoV  clean_rows  pct_clean
N025 trial_00     25      4870         20              6             0              4         0        4840       99.4
N025 trial_01     25      4850          9              0             0              0         0        4841       99.8
N025 trial_02     25      4900         16              0             0              0         0        4884       99.7
N025 trial_03     25      4858         16              5             0              3         0        4834       99.5
N025 trial_04     25      4890         31              2             0              8         0        4849       99.2
N025 trial_05     25      4913         46              7             0              6         0        4854       98.8
N025 trial_06     25      4840         14              0             0              0         0        4826       99.7
N025 trial_07     25      4900         29       

### Per-size cleaning statistics

In [10]:
size_summary = (
    stats_df
    .groupby("size", sort=True)
    .agg(
        trials        = ("trial",         "count"),
        tasks_ok      = ("tasks",         "sum"),
        raw_rows      = ("raw_rows",      "sum"),
        nan_gamma     = ("nan_gamma",     "sum"),
        bad_thick_low = ("bad_thick_low", "sum"),
        bad_thick_hi  = ("bad_thick_hi",  "sum"),
        bad_gamma_low = ("bad_gamma_low", "sum"),
        bad_rhoV      = ("bad_rhoV",      "sum"),
        clean_rows    = ("clean_rows",    "sum"),
    )
)
size_summary["pct_clean"] = (100 * size_summary["clean_rows"] / size_summary["raw_rows"]).round(1)

print(size_summary.to_string())
print()

tot = stats_df[['raw_rows','nan_gamma','bad_thick_low','bad_thick_hi','bad_gamma_low','bad_rhoV','clean_rows']].sum()
print(f"TOTAL")
print(f"  Raw rows                          : {tot['raw_rows']:>8,}")
print(f"  Dropped NaN gamma                 : {tot['nan_gamma']:>8,}")
print(f"  Dropped thickness ≤ 0             : {tot['bad_thick_low']:>8,}")
print(f"  Dropped thickness > {MAX_IFT_THICKNESS} nm        : {tot['bad_thick_hi']:>8,}")
print(f"  Dropped gamma < {MIN_GAMMA} mN/m (near-crit): {tot['bad_gamma_low']:>8,}")
print(f"  Dropped rhoV ≥ {MAX_VAPOR_DENSITY} kg/m3          : {tot['bad_rhoV']:>8,}")
print(f"  Clean rows retained               : {tot['clean_rows']:>8,}  ({100*tot['clean_rows']/tot['raw_rows']:.2f}%)")

      trials  tasks_ok  raw_rows  nan_gamma  bad_thick_low  bad_thick_hi  bad_gamma_low  bad_rhoV  clean_rows  pct_clean
size                                                                                                                    
N025      20       500     97346        374             43             1             69         0       96859       99.5
N050      20      1000    194602        777            105             2            193         0      193525       99.4
N075      20      1500    292019       1029            156             3            234         0      290597       99.5
N100      20      2000    389375       1462            178             6            286         0      387443       99.5

TOTAL
  Raw rows                          :  973,342
  Dropped NaN gamma                 :    3,642
  Dropped thickness ≤ 0             :      482
  Dropped thickness > 7.5 nm        :       12
  Dropped gamma < 0.05 mN/m (near-crit):      782
  Dropped rhoV ≥ 400 kg/m3   

### Task completion summary per size

In [11]:
print(f"{'Size':<8} {'Trials done':>12} {'Tasks OK':>10} {'Failed':>8} {'Missing':>9}")
print("-" * 52)

for size in sorted({s for s,_ in expected}):
    trials_all    = [(size, t) for s,t in expected if s == size]
    trials_done   = sum(1 for k in trials_all if k in successes)
    tasks_ok      = sum(len(v) for k,v in successes.items() if k[0] == size)
    tasks_failed  = sum(len(v) for k,v in failures.items()  if k[0] == size)
    tasks_missing = sum(len(v) for k,v in missing.items()   if k[0] == size)
    print(f"{size:<8} {trials_done:>7}/{len(trials_all):<4} {tasks_ok:>10} {tasks_failed:>8} {tasks_missing:>9}")

Size      Trials done   Tasks OK   Failed   Missing
----------------------------------------------------
N025          20/20          500        0         0
N050          20/20         1000        0         0
N075          20/20         1500        0         0
N100          20/20         2000        0         0


### MLPreprocessing plots — per size
Loads all clean trial CSVs for each size and runs the full 7-plot suite.
Saved to `COMBINED/<SIZE>/PLOTS/`.

In [12]:
sizes = sorted({s for s, _ in expected})
print(f"Generating per-size plots for {len(sizes)} sizes with {min(N_JOBS, len(sizes))} workers...")

msgs = Parallel(n_jobs=min(N_JOBS, len(sizes)), backend="loky")(
    delayed(_plot_one_size)(size, str(COMBINED_DIR), str(THERMOIFT_SRC), Z_COLS, TARGET)
    for size in sizes
)
for m in msgs:
    print(m)

Generating per-size plots for 4 sizes with 4 workers...


N025: 96,859 rows from 20 trials
N050: 193,525 rows from 20 trials
N075: 290,597 rows from 20 trials
N100: 387,443 rows from 20 trials
